In [7]:
import os
import re
import pdfplumber
import pandas as pd
import numpy as np

try:
    from tqdm.notebook import tqdm
except:
    from tqdm import tqdm

pdf_folder = "../pdfs"
output_file_tabla3 = "tabla_3.csv"

pdf_files = sorted([f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")])
pdf_files

['09_abril_2026.pdf',
 '10_abril_2026.pdf',
 '12_diciembre_2025.pdf',
 '13_abril_2026.pdf',
 '13_febrero_2026.pdf',
 '14_abril_2026.pdf',
 '15_abril_2026.pdf',
 '15_diciembre_2025.pdf',
 '16_abril_2026.pdf',
 '16_diciembre_2025.pdf',
 '16_enero_2026.pdf',
 '16_setiembre_2025.pdf',
 '17_abril_2026.pdf',
 '17_diciembre_2025.pdf',
 '17_febrero_2026.pdf',
 '17_noviembre_2025.pdf',
 '17_setiembre_2025.pdf',
 '18_diciembre_2025.pdf',
 '18_febrero_2026.pdf',
 '18_marzo_2026.pdf',
 '18_noviembre_2025.pdf',
 '18_setiembre_2025.pdf',
 '19_diciembre_2025.pdf',
 '19_enero_2026.pdf',
 '19_febrero_2026.pdf',
 '19_marzo_2026.pdf',
 '19_noviembre_2025.pdf',
 '19_setiembre_2025.pdf',
 '20_abril_2026.pdf',
 '20_enero_2026.pdf',
 '20_febrero_2026.pdf',
 '20_marzo_2026.pdf',
 '20_noviembre_2025.pdf',
 '20_octubre_2025.pdf',
 '21_abril_2026.pdf',
 '21_enero_2026.pdf',
 '21_noviembre_2025.pdf',
 '21_octubre_2025.pdf',
 '22_abril_2026.pdf',
 '22_diciembre_2025.pdf',
 '22_enero_2026.pdf',
 '22_octubre_2025.pd

In [8]:
MESES = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12
}

COLUMNAS_TABLA3 = [
    "fecha", "anio", "mes_txt", "mes_num", "dia",
    "ica", "la_libertad", "lima", "pdf_file"
]

PATRON_FECHA_REPORTE = re.compile(r"Fecha\s+(\d{1,2})/(\d{1,2})/(\d{4})", re.IGNORECASE)
PATRON_MES = re.compile(r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)$", re.IGNORECASE)
PATRON_NUM = re.compile(r"^(\d+(?:\.\d+)?|#N/A|-)$", re.IGNORECASE)

def obtener_fecha_reporte(texto):
    m = PATRON_FECHA_REPORTE.search(texto or "")
    if m:
        return tuple(map(int, m.groups()))
    return None, None, None

def convertir_numero(x):
    x = str(x).strip()
    if x in ["", "-", "#N/A", "#n/a"]:
        return np.nan
    return float(x.replace(",", ""))

def anio_de_fila(mes_fila, mes_reporte, anio_reporte):
    if mes_reporte == 1 and mes_fila == 12:
        return anio_reporte - 1
    return anio_reporte

def agrupar_lineas(words, tolerancia=3):
    lineas = {}
    for w in words:
        key = round(w["top"] / tolerancia) * tolerancia
        lineas.setdefault(key, []).append(w)
    
    salida = []
    for _, ws in sorted(lineas.items()):
        ws = sorted(ws, key=lambda x: x["x0"])
        texto = " ".join(w["text"] for w in ws)
        salida.append({
            "top": min(w["top"] for w in ws),
            "bottom": max(w["bottom"] for w in ws),
            "words": ws,
            "texto": texto
        })
    return salida

def encontrar_region_tabla3(lineas):
    idx_titulo = None
    idx_header = None
    idx_fuente = None

    for i, ln in enumerate(lineas):
        txt = ln["texto"].upper()
        if "PRECIO DEL HUEVO DE PRIMERA CALIDAD EN CENTRO DE PRODUCCION" in txt:
            idx_titulo = i
            break

    if idx_titulo is None:
        return None, None, None

    for i in range(idx_titulo + 1, len(lineas)):
        txt = lineas[i]["texto"]
        if all(x in txt for x in ["Fecha", "Ica", "La", "Libertad", "Lima"]):
            idx_header = i
            break

    if idx_header is None:
        return None, None, None

    for i in range(idx_header + 1, len(lineas)):
        txt = lineas[i]["texto"]
        if "Fuente: Empresas productoras de huevo" in txt:
            idx_fuente = i
            break

    if idx_fuente is None:
        idx_fuente = len(lineas)

    return idx_titulo, idx_header, idx_fuente

def centros_columnas_tabla3(header_words):
    centros = {}

    for w in header_words:
        txt = w["text"]
        cx = (w["x0"] + w["x1"]) / 2

        if txt == "Ica":
            centros["ica"] = cx
        elif txt == "Lima":
            centros["lima"] = cx

    la_words = [w for w in header_words if w["text"] in ["La", "Libertad"]]
    if la_words:
        centros["la_libertad"] = np.mean([(w["x0"] + w["x1"]) / 2 for w in la_words])

    return centros

def parsear_tabla3_desde_words(words, texto_pagina, pdf_file):
    _, mes_rep, anio_rep = obtener_fecha_reporte(texto_pagina)

    if anio_rep is None:
        return pd.DataFrame(columns=COLUMNAS_TABLA3)

    lineas = agrupar_lineas(words)
    idx_titulo, idx_header, idx_fuente = encontrar_region_tabla3(lineas)

    if idx_header is None:
        return pd.DataFrame(columns=COLUMNAS_TABLA3)

    centros = centros_columnas_tabla3(lineas[idx_header]["words"])

    if not all(c in centros for c in ["ica", "la_libertad", "lima"]):
        return pd.DataFrame(columns=COLUMNAS_TABLA3)

    filas = []

    for ln in lineas[idx_header + 1:idx_fuente]:
        ws = ln["words"]
        textos = [w["text"] for w in ws]

        if len(textos) < 2:
            continue

        mes_txt = textos[0].lower()
        dia_txt = textos[1]

        if not PATRON_MES.match(mes_txt):
            continue

        if not dia_txt.isdigit():
            continue

        dia = int(dia_txt)
        mes_num = MESES[mes_txt]
        anio_fila = anio_de_fila(mes_num, mes_rep, anio_rep)

        fila = {
            "anio": anio_fila,
            "mes_txt": mes_txt,
            "mes_num": mes_num,
            "dia": dia,
            "ica": np.nan,
            "la_libertad": np.nan,
            "lima": np.nan,
            "pdf_file": pdf_file
        }

        for w in ws[2:]:
            valor_txt = w["text"]

            if not PATRON_NUM.match(valor_txt):
                continue

            cx = (w["x0"] + w["x1"]) / 2

            columna_cercana = min(
                centros.keys(),
                key=lambda c: abs(cx - centros[c])
            )

            fila[columna_cercana] = convertir_numero(valor_txt)

        fecha = pd.to_datetime(
            {"year": [anio_fila], "month": [mes_num], "day": [dia]},
            errors="coerce"
        )[0]

        if pd.isna(fecha):
            continue

        fila["fecha"] = fecha
        filas.append(fila)

    df = pd.DataFrame(filas, columns=COLUMNAS_TABLA3)

    if df.empty:
        return pd.DataFrame(columns=COLUMNAS_TABLA3)

    df = df.drop_duplicates()

    if df["fecha"].duplicated().any():
        cols = ["ica", "la_libertad", "lima"]
        df["n_no_nulos"] = df[cols].notna().sum(axis=1)
        df = (
            df.sort_values(["fecha", "n_no_nulos"], ascending=[True, False])
              .drop_duplicates(subset=["fecha"], keep="first")
              .drop(columns="n_no_nulos")
        )

    return df.sort_values("fecha").reset_index(drop=True)

In [9]:
if os.path.exists(output_file_tabla3):
    df_hist_tabla3 = pd.read_csv(output_file_tabla3, parse_dates=["fecha"])

    # Si el CSV antiguo no tenía pdf_file, lo creamos vacío
    if "pdf_file" not in df_hist_tabla3.columns:
        df_hist_tabla3["pdf_file"] = pd.NA

    # Limpiar filas antiguas sin pdf_file
    df_hist_tabla3 = df_hist_tabla3[df_hist_tabla3["pdf_file"].notna()].copy()

    # Asegurar columnas esperadas
    for col in COLUMNAS_TABLA3:
        if col not in df_hist_tabla3.columns:
            df_hist_tabla3[col] = pd.NA

    df_hist_tabla3 = df_hist_tabla3[COLUMNAS_TABLA3]

    # Guardar histórico limpio
    df_hist_tabla3.to_csv(output_file_tabla3, index=False)

    pdfs_procesados = set(df_hist_tabla3["pdf_file"].dropna().unique())

else:
    df_hist_tabla3 = pd.DataFrame(columns=COLUMNAS_TABLA3)
    pdfs_procesados = set()

pdf_files_nuevos = [f for f in pdf_files if f not in pdfs_procesados]

print("PDFs encontrados:", len(pdf_files))
print("PDFs ya procesados:", len(pdfs_procesados))
print("PDFs nuevos:", len(pdf_files_nuevos))

pdf_files_nuevos[:10]

PDFs encontrados: 84
PDFs ya procesados: 0
PDFs nuevos: 84


['09_abril_2026.pdf',
 '10_abril_2026.pdf',
 '12_diciembre_2025.pdf',
 '13_abril_2026.pdf',
 '13_febrero_2026.pdf',
 '14_abril_2026.pdf',
 '15_abril_2026.pdf',
 '15_diciembre_2025.pdf',
 '16_abril_2026.pdf',
 '16_diciembre_2025.pdf']

In [10]:
registros = []
errores_extraccion = []

for file_name in tqdm(pdf_files_nuevos, desc="Extrayendo tabla 3", unit="pdf"):
    pdf_path = os.path.join(pdf_folder, file_name)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            page = pdf.pages[0]
            texto_pagina = page.extract_text() or ""
            words = page.extract_words()

        registros.append({
            "pdf_file": file_name,
            "texto_pagina": texto_pagina,
            "words": words
        })

    except Exception as e:
        errores_extraccion.append({
            "pdf_file": file_name,
            "error": str(e)
        })

df_extraccion_tabla3 = pd.DataFrame(registros)
df_errores_extraccion_tabla3 = pd.DataFrame(errores_extraccion)

display(df_extraccion_tabla3[["pdf_file"]].head())
display(df_errores_extraccion_tabla3)

Extrayendo tabla 3:   0%|          | 0/84 [00:00<?, ?pdf/s]

,pdf_file
0,09_abril_2026.pdf
1,10_abril_2026.pdf
2,12_diciembre_2025.pdf
3,13_abril_2026.pdf
4,13_febrero_2026.pdf


""


In [11]:
lista_dfs = []
errores_parseo = []

for _, row in df_extraccion_tabla3.iterrows():
    try:
        df_tmp = parsear_tabla3_desde_words(
            words=row["words"],
            texto_pagina=row["texto_pagina"],
            pdf_file=row["pdf_file"]
        )
        lista_dfs.append(df_tmp)

    except Exception as e:
        errores_parseo.append({
            "pdf_file": row["pdf_file"],
            "error": str(e)
        })

df_nuevos_tabla3 = (
    pd.concat(lista_dfs, ignore_index=True)
    if lista_dfs else pd.DataFrame(columns=COLUMNAS_TABLA3)
)

df_errores_parseo_tabla3 = pd.DataFrame(errores_parseo)

df_total_tabla3 = pd.concat([df_hist_tabla3, df_nuevos_tabla3], ignore_index=True)
df_total_tabla3 = df_total_tabla3.drop_duplicates()

if not df_total_tabla3.empty:
    df_total_tabla3 = (
        df_total_tabla3
        .drop_duplicates(subset=["fecha", "pdf_file"], keep="first")
        .sort_values(["fecha", "pdf_file"])
        .reset_index(drop=True)
    )

df_total_tabla3.to_csv(output_file_tabla3, index=False)

print("Guardado en:", output_file_tabla3)
print("Filas nuevas:", len(df_nuevos_tabla3))
print("Filas totales:", len(df_total_tabla3))

display(df_nuevos_tabla3.head())
display(df_errores_parseo_tabla3)

Guardado en: tabla_3.csv
Filas nuevas: 1344
Filas totales: 1344


,fecha,anio,mes_txt,mes_num,dia,ica,la_libertad,lima,pdf_file
0,2026-03-24,2026,mar,3,24,5.97,5.3,6.17,09_abril_2026.pdf
1,2026-03-25,2026,mar,3,25,5.99,5.2,5.56,09_abril_2026.pdf
2,2026-03-26,2026,mar,3,26,5.89,5.2,6.07,09_abril_2026.pdf
3,2026-03-27,2026,mar,3,27,5.92,NaN,6.00,09_abril_2026.pdf
4,2026-03-28,2026,mar,3,28,6.33,NaN,5.97,09_abril_2026.pdf


""


In [12]:
cols_num = ["ica", "la_libertad", "lima"]

if df_total_tabla3.empty:
    print("No se encontraron datos para tabla 3.")
else:
    duplicados_fecha = df_total_tabla3[
        df_total_tabla3.duplicated(subset=["fecha"], keep=False)
    ].sort_values("fecha")

    filas_todo_nan = df_total_tabla3[
        df_total_tabla3[cols_num].isna().all(axis=1)
    ]

    display(duplicados_fecha)
    display(filas_todo_nan)

,fecha,anio,mes_txt,mes_num,dia,ica,la_libertad,lima,pdf_file
1,2025-09-01,2025,sep,9,1,5.47,5.00,5.9,16_setiembre_2025.pdf
2,2025-09-01,2025,sep,9,1,5.47,5.00,5.9,17_setiembre_2025.pdf
3,2025-09-02,2025,sep,9,2,5.43,5.00,5.9,16_setiembre_2025.pdf
4,2025-09-02,2025,sep,9,2,5.43,5.00,5.9,17_setiembre_2025.pdf
5,2025-09-02,2025,sep,9,2,5.43,5.00,5.9,18_setiembre_2025.pdf
...,...,...,...,...,...,...,...,...,...
1338,2026-04-20,2026,abr,4,20,6.01,NaN,5.8,23_abril_2026.pdf
1339,2026-04-20,2026,abr,4,20,6.01,NaN,5.8,26_abril_2026.pdf
1341,2026-04-21,2026,abr,4,21,5.88,5.25,5.8,23_abril_2026.pdf
1340,2026-04-21,2026,abr,4,21,5.88,5.25,5.8,22_abril_2026.pdf


,fecha,anio,mes_txt,mes_num,dia,ica,la_libertad,lima,pdf_file


In [13]:
df_total_tabla3

,fecha,anio,mes_txt,mes_num,dia,ica,la_libertad,lima,pdf_file
0,2025-08-31,2025,ago,8,31,5.15,NaN,5.90,16_setiembre_2025.pdf
1,2025-09-01,2025,sep,9,1,5.47,5.00,5.90,16_setiembre_2025.pdf
2,2025-09-01,2025,sep,9,1,5.47,5.00,5.90,17_setiembre_2025.pdf
3,2025-09-02,2025,sep,9,2,5.43,5.00,5.90,16_setiembre_2025.pdf
4,2025-09-02,2025,sep,9,2,5.43,5.00,5.90,17_setiembre_2025.pdf
...,...,...,...,...,...,...,...,...,...
1339,2026-04-20,2026,abr,4,20,6.01,NaN,5.80,26_abril_2026.pdf
1340,2026-04-21,2026,abr,4,21,5.88,5.25,5.80,22_abril_2026.pdf
1341,2026-04-21,2026,abr,4,21,5.88,5.25,5.80,23_abril_2026.pdf
1342,2026-04-21,2026,abr,4,21,5.88,5.25,5.80,26_abril_2026.pdf


## A veces datos de el pdf se corrigen en días posteriores, entonces para la misma fecha tomamos el último pdf

In [14]:
df_total = df_total_tabla3.copy()

In [15]:
# CREAMOS FECHA pdf
import pandas as pd

# Diccionario de meses
meses_map = {
    'enero':1, 'febrero':2, 'marzo':3, 'abril':4,
    'mayo':5, 'junio':6, 'julio':7, 'agosto':8,
    'setiembre':9, 'septiembre':9, 'octubre':10,
    'noviembre':11, 'diciembre':12
}

# Extraer partes del nombre del PDF
df_total[['dia_pdf', 'mes_pdf_txt', 'anio_pdf']] = df_total['pdf_file'] \
    .str.replace('.pdf', '', regex=False) \
    .str.split('_', expand=True)

# Convertir mes a número
df_total['mes_pdf'] = df_total['mes_pdf_txt'].map(meses_map)

# Crear fecha_pdf
df_total['fecha_pdf'] = pd.to_datetime(
    dict(year=df_total['anio_pdf'].astype(int),
         month=df_total['mes_pdf'],
         day=df_total['dia_pdf'].astype(int))
)

In [16]:
# p sea la fecha las ponemos en orden,  y leugo ordenamos por fecha pdf, la más reciente (primera o última ) es la que queda
df_total= df_total.sort_values(['fecha', 'fecha_pdf']) # ordena en ascendente por defecto (la íultima es la mas reciente)

df_final = df_total.drop_duplicates(subset=['fecha'], keep='last')

In [17]:
df_final.shape

(227, 14)

In [18]:
df_final

,fecha,anio,mes_txt,mes_num,dia,ica,la_libertad,lima,pdf_file,dia_pdf,mes_pdf_txt,anio_pdf,mes_pdf,fecha_pdf
0,2025-08-31,2025,ago,8,31,5.15,NaN,5.90,16_setiembre_2025.pdf,16,setiembre,2025,9,2025-09-16
2,2025-09-01,2025,sep,9,1,5.47,5.00,5.90,17_setiembre_2025.pdf,17,setiembre,2025,9,2025-09-17
5,2025-09-02,2025,sep,9,2,5.43,5.00,5.90,18_setiembre_2025.pdf,18,setiembre,2025,9,2025-09-18
9,2025-09-03,2025,sep,9,3,5.41,4.90,5.50,19_setiembre_2025.pdf,19,setiembre,2025,9,2025-09-19
13,2025-09-04,2025,sep,9,4,5.31,4.80,5.50,19_setiembre_2025.pdf,19,setiembre,2025,9,2025-09-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1330,2026-04-18,2026,abr,4,18,6.39,NaN,6.48,26_abril_2026.pdf,26,abril,2026,4,2026-04-26
1335,2026-04-19,2026,abr,4,19,5.61,NaN,6.48,26_abril_2026.pdf,26,abril,2026,4,2026-04-26
1339,2026-04-20,2026,abr,4,20,6.01,NaN,5.80,26_abril_2026.pdf,26,abril,2026,4,2026-04-26
1342,2026-04-21,2026,abr,4,21,5.88,5.25,5.80,26_abril_2026.pdf,26,abril,2026,4,2026-04-26


In [20]:
# eliminamos pdf_file porque no nos aporta nada
df_final = df_final.drop(columns=['pdf_file', 'dia_pdf', 'mes_pdf_txt', 'anio_pdf', 'mes_pdf'])

In [22]:
df_final = df_final.drop(columns=['fecha_pdf'])

In [23]:
df_final

,fecha,anio,mes_txt,mes_num,dia,ica,la_libertad,lima
0,2025-08-31,2025,ago,8,31,5.15,NaN,5.90
2,2025-09-01,2025,sep,9,1,5.47,5.00,5.90
5,2025-09-02,2025,sep,9,2,5.43,5.00,5.90
9,2025-09-03,2025,sep,9,3,5.41,4.90,5.50
13,2025-09-04,2025,sep,9,4,5.31,4.80,5.50
...,...,...,...,...,...,...,...,...
1330,2026-04-18,2026,abr,4,18,6.39,NaN,6.48
1335,2026-04-19,2026,abr,4,19,5.61,NaN,6.48
1339,2026-04-20,2026,abr,4,20,6.01,NaN,5.80
1342,2026-04-21,2026,abr,4,21,5.88,5.25,5.80


In [24]:
df_final.duplicated(subset=['fecha']).any()

False

In [25]:
# finalmente descargamos en csv
df_final.to_csv("tabla_3.csv", index=False)